# Spark Analysis — AirQuality Alert (Kelompok 6)

Membaca data dari **HDFS** (`/data/airquality/api/` + `/data/airquality/rss/`) dan menghasilkan 3 analisis wajib:

1. **Distribusi kategori AQI per kota (%)** — Baik / Sedang / Tidak Sehat / Berbahaya
2. **Rata-rata AQI per kota per jam** — pakai **Spark SQL**, untuk identifikasi jam puncak polusi
3. **Ranking kota AQI terburuk** — urut rata-rata AQI ↓ + hitung jumlah event "Tidak Sehat+"

Output:
- File parquet ke `hdfs:///data/airquality/hasil/` (3 folder)
- File ringkasan JSON ke `dashboard/data/spark_results.json` untuk dashboard

In [ ]:
import json
import os
import shutil
import tempfile
from datetime import datetime, timezone
from pathlib import Path

REPO_ROOT = Path(os.getcwd()).resolve()
if REPO_ROOT.name == "spark":
    REPO_ROOT = REPO_ROOT.parent
DASHBOARD_DATA_DIR = REPO_ROOT / "dashboard" / "data"
DASHBOARD_DATA_DIR.mkdir(parents=True, exist_ok=True)

HDFS_NAMENODE_RPC = os.environ.get("HDFS_NAMENODE_RPC", "hdfs://localhost:9000")
HDFS_NAMENODE_WEB = os.environ.get("HDFS_NAMENODE_URL", "http://localhost:9870")
HDFS_USER = os.environ.get("HDFS_USER", "root")
HDFS_BASE = "/data/airquality"

print("Repo root :", REPO_ROOT)
print("HDFS RPC  :", HDFS_NAMENODE_RPC)
print("HDFS Web  :", HDFS_NAMENODE_WEB)

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

spark = (
    SparkSession.builder.appName("AirQualityAnalysis")
    .config("spark.sql.session.timeZone", "Asia/Jakarta")
    .config("spark.hadoop.fs.defaultFS", HDFS_NAMENODE_RPC)
    .config("spark.hadoop.dfs.client.use.datanode.hostname", "true")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)

## 1. Helper baca HDFS

Strategi: coba `spark.read.json("hdfs://...")` langsung. Jika gagal (umumnya WSL host
tidak bisa resolve hostname `datanode1`), fallback download via WebHDFS (library `hdfs`)
ke folder lokal sementara, lalu Spark baca dari sana.

Sumber data tetap **HDFS** — fallback hanya cara mengangkut byte ke driver Spark.

In [3]:
import subprocess
from hdfs import InsecureClient

hdfs_client = InsecureClient(HDFS_NAMENODE_WEB, user=HDFS_USER)
HDFS_NAMENODE_CONTAINER = os.environ.get("HDFS_NAMENODE_CONTAINER", "namenode")


def _list_hdfs(subdir: str):
    """List nama file .json di {HDFS_BASE}/{subdir} via WebHDFS (NameNode-only ⇒ aman di WSL2)."""
    try:
        return [n for n in hdfs_client.list(f"{HDFS_BASE}/{subdir}") if n.endswith(".json")]
    except Exception as exc:
        print(f"[WARN] WebHDFS list gagal untuk {subdir}: {exc}")
        return []


def _cat_via_docker(hdfs_path: str) -> bytes:
    """Ambil isi file dari HDFS via `docker exec namenode hdfs dfs -cat`.
    Cara ini robust di WSL2 karena seluruh komunikasi NameNode↔DataNode tetap
    di dalam jaringan Docker (tidak pernah keluar ke host)."""
    proc = subprocess.run(
        ["docker", "exec", HDFS_NAMENODE_CONTAINER, "hdfs", "dfs", "-cat", hdfs_path],
        capture_output=True, timeout=120,
    )
    if proc.returncode != 0:
        raise RuntimeError(f"docker hdfs cat gagal: {proc.stderr.decode('utf-8', 'replace')}")
    return proc.stdout


def read_json_from_hdfs(subdir: str):
    """Return Spark DataFrame dari semua file *.json di {HDFS_BASE}/{subdir}/.
    Strategi:
      1. Coba `spark.read.json(hdfs://...)` native.
      2. Kalau gagal (umum di WSL2: namenode redirect ke datanode hostname yang
         tidak resolvable), fallback: list pakai WebHDFS (NameNode-only),
         download isi tiap file via `docker exec namenode hdfs dfs -cat`,
         lalu Spark baca dari folder lokal sementara.
    Sumber data tetap **HDFS** — fallback hanya jembatan transport ke driver Spark.
    """
    hdfs_path = f"{HDFS_NAMENODE_RPC}{HDFS_BASE}/{subdir}/"
    try:
        df = spark.read.option("multiLine", False).json(hdfs_path)
        if df.head(1):
            print(f"[OK ] Native HDFS read berhasil untuk {subdir}")
            return df
    except Exception as exc:
        print(f"[WARN] Native HDFS read gagal untuk {subdir}: {type(exc).__name__}")

    print(f"[INFO] Fallback `docker exec hdfs dfs -cat` untuk {subdir} ...")
    files = _list_hdfs(subdir)
    if not files:
        print(f"[WARN] Tidak ada file .json di HDFS {HDFS_BASE}/{subdir}/")
        return spark.createDataFrame([], schema="city string")
    tmp_dir = tempfile.mkdtemp(prefix=f"airquality_{subdir}_")
    for fname in files:
        content = _cat_via_docker(f"{HDFS_BASE}/{subdir}/{fname}")
        with open(os.path.join(tmp_dir, fname), "wb") as fout:
            fout.write(content)
    df = spark.read.option("multiLine", False).json(tmp_dir)
    print(f"[OK ] {len(files)} file dari HDFS di-download via docker, di-read oleh Spark.")
    return df

ModuleNotFoundError: No module named 'hdfs'

In [ ]:
df_api_raw = read_json_from_hdfs("api")
df_api_raw.printSchema()
print("Total record API :", df_api_raw.count())
df_api_raw.show(5, truncate=False)

In [ ]:
# Normalisasi: pastikan kolom yang dipakai analisis selalu ada bahkan saat data masih sedikit.
needed_cols = {
    "city": F.lit(None).cast("string"),
    "aqi": F.lit(None).cast("double"),
    "timestamp_ingest": F.lit(None).cast("string"),
    "observed_at": F.lit(None).cast("string"),
}
for cname, fallback in needed_cols.items():
    if cname not in df_api_raw.columns:
        df_api_raw = df_api_raw.withColumn(cname, fallback)

df_api = (
    df_api_raw
    .withColumn("city", F.coalesce(F.col("city"), F.lit("unknown")))
    .withColumn("aqi", F.col("aqi").cast("double"))
    .withColumn(
        "event_ts",
        F.coalesce(
            F.to_timestamp("observed_at"),
            F.to_timestamp("timestamp_ingest"),
        ),
    )
    .withColumn("hour", F.hour("event_ts"))
    .withColumn(
        "category",
        F.when(F.col("aqi") <= 50, "Baik")
        .when(F.col("aqi") <= 100, "Sedang")
        .when(F.col("aqi") <= 150, "Tidak Sehat")
        .otherwise("Berbahaya"),
    )
    .filter(F.col("aqi").isNotNull())
)
df_api.createOrReplaceTempView("airquality")
df_api.cache()
print("Total record valid:", df_api.count())
df_api.select("city", "aqi", "category", "event_ts", "hour").show(8, truncate=False)

## Analisis 1 — Distribusi Kategori AQI per Kota (%)

Klasifikasi event tiap kota ke 4 kategori; output persentase per kategori per kota.

In [ ]:
from pyspark.sql.window import Window

win_city = Window.partitionBy("city")

df_dist = (
    df_api.groupBy("city", "category").agg(F.count("*").alias("n"))
    .withColumn("total", F.sum("n").over(win_city))
    .withColumn("pct", F.round(F.col("n") / F.col("total") * 100, 2))
)

df_dist_pivot = (
    df_dist.groupBy("city")
    .pivot("category", ["Baik", "Sedang", "Tidak Sehat", "Berbahaya"])
    .agg(F.first("pct"))
    .na.fill(0.0)
    .withColumnRenamed("Baik", "baik_pct")
    .withColumnRenamed("Sedang", "sedang_pct")
    .withColumnRenamed("Tidak Sehat", "tidak_sehat_pct")
    .withColumnRenamed("Berbahaya", "berbahaya_pct")
    .orderBy("city")
)
df_dist_pivot.show(truncate=False)

**Interpretasi:** persentase di atas menunjukkan komposisi kualitas udara setiap kota selama periode pengamatan. Kota dengan persentase **Tidak Sehat + Berbahaya** lebih tinggi adalah prioritas peringatan dari Dinas Kesehatan.

## Analisis 2 — Rata-rata AQI per Kota per Jam (Spark SQL)

Identifikasi jam-jam puncak polusi. Kita pakai **Spark SQL murni** lewat `spark.sql(...)`.

In [ ]:
df_avg_hour = spark.sql(
    """
    SELECT city,
           hour,
           ROUND(AVG(aqi), 2) AS avg_aqi,
           COUNT(*)           AS n_event
    FROM airquality
    WHERE hour IS NOT NULL
    GROUP BY city, hour
    ORDER BY city, hour
    """
)
df_avg_hour.show(50, truncate=False)

# Top 3 jam puncak per kota (untuk diringkas ke dashboard)
df_peak = spark.sql(
    """
    WITH hourly AS (
      SELECT city, hour, AVG(aqi) AS avg_aqi
      FROM airquality
      WHERE hour IS NOT NULL
      GROUP BY city, hour
    ),
    ranked AS (
      SELECT city, hour, avg_aqi,
             ROW_NUMBER() OVER (PARTITION BY city ORDER BY avg_aqi DESC) AS rn
      FROM hourly
    )
    SELECT city, hour, ROUND(avg_aqi, 2) AS avg_aqi
    FROM ranked
    WHERE rn <= 3
    ORDER BY city, avg_aqi DESC
    """
)
df_peak.show(20, truncate=False)

**Interpretasi:** baris pertama di setiap kota merupakan **jam puncak polusi** kota tersebut. Pola yang umum: pagi hari (07:00–09:00) saat traffic puncak, dan sore (17:00–19:00). Dinas Kesehatan dapat menjadwalkan peringatan otomatis pada jam-jam ini.

## Analisis 3 — Ranking Kota AQI Terburuk + Jumlah Event Tidak Sehat+

In [ ]:
df_rank = spark.sql(
    """
    SELECT city,
           ROUND(AVG(aqi), 2) AS avg_aqi,
           SUM(CASE WHEN aqi > 100 THEN 1 ELSE 0 END) AS unhealthy_or_worse_events,
           COUNT(*) AS total_events
    FROM airquality
    GROUP BY city
    ORDER BY avg_aqi DESC
    """
)
df_rank.show(truncate=False)

**Interpretasi:** kota teratas adalah kandidat utama intervensi Dinas Kesehatan. Kolom `unhealthy_or_worse_events` menunjukkan berapa kali AQI kota tersebut menyentuh kategori **Tidak Sehat (>100)** atau lebih buruk — ini metrik frekuensi gangguan kesehatan.

## Simpan hasil

1. Simpan tiap analisis sebagai parquet di **HDFS** (`/data/airquality/hasil/...`).
2. Simpan ringkasan terstruktur sebagai **`dashboard/data/spark_results.json`** sehingga Flask dashboard bisa membacanya.

In [ ]:
def save_to_hdfs_parquet(df, name: str):
    """Simpan DataFrame ke HDFS sebagai parquet.
    Strategi:
      1. Coba `df.write.parquet(hdfs://...)` native.
      2. Kalau gagal (WSL2 datanode hostname), fallback: tulis parquet ke folder
         lokal lalu `docker cp` + `docker exec namenode hdfs dfs -put` ke HDFS.
    """
    target = f"{HDFS_NAMENODE_RPC}{HDFS_BASE}/hasil/{name}"
    try:
        df.coalesce(1).write.mode("overwrite").parquet(target)
        print(f"[OK ] native parquet → {target}")
        return
    except Exception as exc:
        print(f"[WARN] Native parquet write gagal ({type(exc).__name__}); fallback docker cp.")

    local_dir = tempfile.mkdtemp(prefix=f"hasil_{name}_")
    try:
        df.coalesce(1).write.mode("overwrite").parquet(f"file://{local_dir}")
        hdfs_dir = f"{HDFS_BASE}/hasil/{name}"
        subprocess.run(
            ["docker", "exec", HDFS_NAMENODE_CONTAINER, "hdfs", "dfs", "-rm", "-r", "-f", hdfs_dir],
            capture_output=True,
        )
        subprocess.run(
            ["docker", "exec", HDFS_NAMENODE_CONTAINER, "hdfs", "dfs", "-mkdir", "-p", hdfs_dir],
            capture_output=True, check=True,
        )
        container_tmp = f"/tmp/{name}_parquet"
        subprocess.run(
            ["docker", "exec", HDFS_NAMENODE_CONTAINER, "rm", "-rf", container_tmp],
            capture_output=True,
        )
        subprocess.run(
            ["docker", "cp", local_dir + "/.", f"{HDFS_NAMENODE_CONTAINER}:{container_tmp}"],
            check=True, capture_output=True,
        )
        list_proc = subprocess.run(
            ["docker", "exec", HDFS_NAMENODE_CONTAINER, "ls", container_tmp],
            capture_output=True, check=True,
        )
        for fname in list_proc.stdout.decode().split():
            if fname.startswith("_") or fname.startswith("."):
                continue
            subprocess.run(
                ["docker", "exec", HDFS_NAMENODE_CONTAINER, "hdfs", "dfs", "-put", "-f",
                 f"{container_tmp}/{fname}", f"{hdfs_dir}/{fname}"],
                check=True, capture_output=True,
            )
        subprocess.run(
            ["docker", "exec", HDFS_NAMENODE_CONTAINER, "rm", "-rf", container_tmp],
            capture_output=True,
        )
        print(f"[OK ] parquet via docker → hdfs:{hdfs_dir}")
    finally:
        shutil.rmtree(local_dir, ignore_errors=True)


save_to_hdfs_parquet(df_dist_pivot, "distribusi_kategori")
save_to_hdfs_parquet(df_avg_hour, "avg_aqi_per_jam")
save_to_hdfs_parquet(df_rank, "ranking_kota")

In [ ]:
# Build dashboard/data/spark_results.json (struktur yang dibaca dashboard)
rows_dist = [r.asDict() for r in df_dist_pivot.collect()]
rows_peak = [r.asDict() for r in df_peak.collect()]
rows_rank = [r.asDict() for r in df_rank.collect()]

chart_avg_hour = [
    {"label": f"{r['city']} {int(r['hour']):02d}:00", "avg_aqi": float(r["avg_aqi"]) if r["avg_aqi"] is not None else 0.0}
    for r in rows_peak
]

spark_results = {
    "generated_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "analysis1": {
        "title": "1. Distribusi kategori AQI per kota (%)",
        "interpretation": (
            "Persentase setiap kategori kualitas udara per kota selama periode pengamatan. "
            "Kota dengan total Tidak Sehat + Berbahaya lebih besar perlu prioritas peringatan."
        ),
        "rows": rows_dist,
    },
    "analysis2": {
        "title": "2. Rata-rata AQI per kota per jam (Spark SQL)",
        "interpretation": (
            "Menampilkan 3 jam puncak polusi tiap kota. Pola umumnya muncul di pagi (07-09) dan sore (17-19)."
        ),
        "chart": chart_avg_hour,
    },
    "analysis3": {
        "title": "3. Ranking kota AQI terburuk + event Tidak Sehat+",
        "interpretation": (
            "Kota di peringkat atas memiliki rata-rata AQI tertinggi dan paling sering masuk kategori Tidak Sehat (AQI > 100)."
        ),
        "rows": rows_rank,
    },
    "_demo": False,
}

out_path = DASHBOARD_DATA_DIR / "spark_results.json"
tmp = out_path.with_suffix(".tmp")
with tmp.open("w", encoding="utf-8") as f:
    json.dump(spark_results, f, ensure_ascii=False, indent=2, default=str)
os.replace(tmp, out_path)
print("OK: tersimpan ke", out_path)
print("Jumlah row analysis1:", len(rows_dist))
print("Jumlah row analysis2:", len(chart_avg_hour))
print("Jumlah row analysis3:", len(rows_rank))

In [ ]:
spark.stop()
print("Spark session ditutup. Refresh dashboard untuk melihat hasil analisis nyata.")